# AlpCAN CT Pipeline: Nodül Segmentasyon Eğitimi

**Amaç:** LIDC-IDRI veri seti üzerinde U-Net tabanlı nodül segmentasyon modeli eğitmek.
AlpCAN CT Pipeline Agent 2 (Nodül Tespit) için ağırlık üretimi.

**İçindekiler:**
1. LIDC-IDRI veri seti yükleme ve konsensüs maskeleri
2. Eğitim/doğrulama bölmesi (hasta bazlı)
3. U-Net mimari tanımlama (ResNet-34 encoder)
4. Veri artırma (augmentation)
5. Eğitim (Dice + BCE loss, 50 epoch)
6. Doğrulama metrikleri (Dice, IoU, Precision, Recall)
7. Connected component analizi ile nodül çıkarımı
8. Sonuçları kaydetme + ağırlık export

**Model:** U-Net (ResNet-34 encoder, ImageNet pre-trained)
**GPU:** Kaggle T4 16GB
**Dataset:** `zhangweiled/lidcidri` — LIDC-IDRI kesilmiş nodül dilimleri

In [ ]:
!pip install -q segmentation-models-pytorch albumentations

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import GroupKFold

print(f"PyTorch: {torch.__version__}")
print(f"SMP: {smp.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Bellek: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## 1. LIDC-IDRI Veri Seti Yükleme

In [ ]:
# Dataset yolunu bul
INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# LIDC-IDRI dataset
DATA_DIR = Path("/kaggle/input/lidcidri/LIDC-IDRI-slices")
if not DATA_DIR.exists():
    # Alternatif yol ara
    for candidate in INPUT_DIR.rglob("LIDC-IDRI-0001"):
        DATA_DIR = candidate.parent
        break
    if not DATA_DIR.exists():
        raise FileNotFoundError("LIDC-IDRI dataset bulunamadi! 'zhangweiled/lidcidri' dataset'ini ekleyin.")

print(f"Dataset: {DATA_DIR}")

# Tum hasta ve nodul bilgilerini topla
patient_dirs = sorted([d for d in DATA_DIR.iterdir() if d.is_dir()])
print(f"Toplam hasta: {len(patient_dirs)}")

# Her nodul icin dilim ve maske bilgisi
all_samples = []
for pdir in patient_dirs:
    nodule_dirs = sorted([d for d in pdir.iterdir() if d.is_dir()])
    for ndir in nodule_dirs:
        img_dir = ndir / "images"
        mask_dirs = sorted([d for d in ndir.iterdir() if d.is_dir() and d.name.startswith("mask")])
        
        if not img_dir.exists() or not mask_dirs:
            continue
        
        for img_file in sorted(img_dir.glob("*.png")):
            # Konsensus maskesi: tum annotatorlerin ortalaması >= 0.5
            masks_exist = []
            for mdir in mask_dirs:
                mask_file = mdir / img_file.name
                if mask_file.exists():
                    masks_exist.append(mask_file)
            
            if masks_exist:
                all_samples.append({
                    'patient_id': pdir.name,
                    'nodule': ndir.name,
                    'image_path': str(img_file),
                    'mask_paths': [str(m) for m in masks_exist],
                    'n_annotators': len(masks_exist),
                })

df = pd.DataFrame(all_samples)
print(f"\nToplam dilim (egitim ornegi): {len(df):,}")
print(f"Toplam hasta: {df['patient_id'].nunique()}")
print(f"Toplam nodul: {df.groupby(['patient_id', 'nodule']).ngroups}")
print(f"Annotator dagilimi:\n{df['n_annotators'].value_counts().sort_index()}")

---
## 2. Eğitim/Doğrulama Bölmesi (Hasta Bazlı)

In [ ]:
# Hasta bazli 80/20 bolme (veri sizintisi onleme)
np.random.seed(42)
patients = df['patient_id'].unique()
np.random.shuffle(patients)

split_idx = int(len(patients) * 0.8)
train_patients = set(patients[:split_idx])
val_patients = set(patients[split_idx:])

train_df = df[df['patient_id'].isin(train_patients)].reset_index(drop=True)
val_df = df[df['patient_id'].isin(val_patients)].reset_index(drop=True)

print(f"Train: {len(train_df):,} dilim ({len(train_patients)} hasta)")
print(f"Val:   {len(val_df):,} dilim ({len(val_patients)} hasta)")

---
## 3. Dataset ve Augmentation

In [ ]:
IMG_SIZE = 256

# Augmentation
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=30, p=0.5),
    A.ElasticTransform(alpha=120, sigma=120 * 0.05, p=0.2),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])


class LIDCSegDataset(Dataset):
    """LIDC-IDRI nodul segmentasyon dataset'i.
    
    Konsensus maskesi: Tum annotator maskelerinin ortalamasi >= 0.5 threshold.
    Goruntu 3 kanala kopyalanir (grayscale -> RGB, ImageNet pretrained encoder icin).
    """
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Goruntu oku (grayscale -> 3 kanal)
        img = np.array(Image.open(row['image_path']).convert('L'))
        img_rgb = np.stack([img, img, img], axis=-1)  # (H, W, 3)
        
        # Konsensus maskesi olustur
        mask_sum = np.zeros(img.shape[:2], dtype=np.float32)
        n_masks = 0
        for mask_path in row['mask_paths']:
            m = np.array(Image.open(mask_path).convert('L'))
            mask_sum += (m > 0).astype(np.float32)
            n_masks += 1
        
        consensus = (mask_sum / max(n_masks, 1) >= 0.5).astype(np.float32)
        
        if self.transform:
            augmented = self.transform(image=img_rgb, mask=consensus)
            img_tensor = augmented['image']  # (3, H, W)
            mask_tensor = augmented['mask'].unsqueeze(0)  # (1, H, W)
        else:
            img_tensor = torch.from_numpy(img_rgb.transpose(2, 0, 1)).float()
            mask_tensor = torch.from_numpy(consensus).unsqueeze(0).float()
        
        return img_tensor, mask_tensor


# Dataset olustur
train_dataset = LIDCSegDataset(train_df, transform=train_transform)
val_dataset = LIDCSegDataset(val_df, transform=val_transform)

BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train dataset: {len(train_dataset)} ornek")
print(f"Val dataset: {len(val_dataset)} ornek")
print(f"Batch size: {BATCH_SIZE}")

# Ornek gorselleştirme
img_sample, mask_sample = train_dataset[0]
print(f"Goruntu shape: {img_sample.shape}")
print(f"Maske shape: {mask_sample.shape}")

In [ ]:
# Ornek batch gorsellestirme
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Egitim Ornekleri — Goruntu + Konsensus Maske', fontsize=14)

for i in range(4):
    img, mask = val_dataset[i * 10]  # Her 10. ornek
    
    # Denormalize
    img_np = img.numpy().transpose(1, 2, 0)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_np = (img_np * std + mean).clip(0, 1)
    
    axes[0, i].imshow(img_np[:, :, 0], cmap='gray')
    axes[0, i].set_title(f'Dilim {i*10}')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(img_np[:, :, 0], cmap='gray')
    mask_np = mask.numpy()[0]
    if mask_np.max() > 0:
        axes[1, i].contour(mask_np, colors='lime', linewidths=1.5, levels=[0.5])
    axes[1, i].set_title(f'+ Maske')
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ct_seg_training_samples.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Model Tanımlama — U-Net (ResNet-34 Encoder)

In [ ]:
# U-Net modeli — ResNet-34 encoder (ImageNet pre-trained)
model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,  # Binary segmentasyon (nodul / arkaplan)
    activation=None,  # Raw logits — loss icinde sigmoid uygulanacak
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: U-Net (ResNet-34 encoder)")
print(f"Toplam parametre: {n_params:,} ({n_params/1e6:.1f}M)")
print(f"Egitilecek: {n_trainable:,}")
print(f"Device: {device}")

---
## 5. Loss Fonksiyonu ve Optimizer

In [ ]:
class DiceBCELoss(nn.Module):
    """Dice Loss + BCE Loss kombinasyonu.
    
    Dice: Bolgel overlap metrik (class imbalance'a dayanikli)
    BCE: Piksel bazli binary cross-entropy
    """
    def __init__(self, dice_weight=0.5, bce_weight=0.5, smooth=1.0):
        super().__init__()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()
    
    def dice_loss(self, pred, target):
        pred_sigmoid = torch.sigmoid(pred)
        intersection = (pred_sigmoid * target).sum(dim=(2, 3))
        union = pred_sigmoid.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dice.mean()
    
    def forward(self, pred, target):
        return self.dice_weight * self.dice_loss(pred, target) + self.bce_weight * self.bce(pred, target)


criterion = DiceBCELoss(dice_weight=0.5, bce_weight=0.5)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)

print("Loss: DiceBCE (0.5 Dice + 0.5 BCE)")
print("Optimizer: AdamW (lr=1e-4, wd=1e-4)")
print("Scheduler: CosineAnnealing (T_max=50)")

---
## 6. Eğitim Döngüsü

In [ ]:
def compute_metrics(pred, target, threshold=0.5):
    """Segmentasyon metrikleri hesapla."""
    pred_binary = (torch.sigmoid(pred) > threshold).float()
    
    intersection = (pred_binary * target).sum(dim=(2, 3))
    union = pred_binary.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    pred_sum = pred_binary.sum(dim=(2, 3))
    target_sum = target.sum(dim=(2, 3))
    
    smooth = 1e-6
    dice = (2.0 * intersection + smooth) / (union + smooth)
    iou = (intersection + smooth) / (union - intersection + smooth)
    precision = (intersection + smooth) / (pred_sum + smooth)
    recall = (intersection + smooth) / (target_sum + smooth)
    
    return {
        'dice': dice.mean().item(),
        'iou': iou.mean().item(),
        'precision': precision.mean().item(),
        'recall': recall.mean().item(),
    }


NUM_EPOCHS = 50
best_val_dice = 0.0
history = {'train_loss': [], 'val_loss': [], 'val_dice': [], 'val_iou': [], 'lr': []}
scaler = torch.amp.GradScaler('cuda')

print(f"Egitim basliyor — {NUM_EPOCHS} epoch")
print("=" * 70)

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    # --- Train ---
    model.train()
    train_loss = 0.0
    n_train = 0
    
    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, masks)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        train_loss += loss.item() * images.size(0)
        n_train += images.size(0)
    
    train_loss /= n_train
    
    # --- Validation ---
    model.eval()
    val_loss = 0.0
    val_metrics = defaultdict(float)
    n_val = 0
    
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)
            
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, masks)
            
            val_loss += loss.item() * images.size(0)
            metrics = compute_metrics(outputs, masks)
            for k, v in metrics.items():
                val_metrics[k] += v * images.size(0)
            n_val += images.size(0)
    
    val_loss /= n_val
    for k in val_metrics:
        val_metrics[k] /= n_val
    
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    
    # History kaydet
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_dice'].append(val_metrics['dice'])
    history['val_iou'].append(val_metrics['iou'])
    history['lr'].append(current_lr)
    
    # Best model kaydet
    if val_metrics['dice'] > best_val_dice:
        best_val_dice = val_metrics['dice']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_dice': best_val_dice,
            'val_iou': val_metrics['iou'],
        }, OUTPUT_DIR / 'ct_seg_best_model.pth')
        marker = ' << BEST'
    else:
        marker = ''
    
    # Her 5 epoch'ta rapor
    if (epoch + 1) % 5 == 0 or epoch == 0:
        elapsed = time.time() - start_time
        print(f"Epoch {epoch+1:>3d}/{NUM_EPOCHS} | "
              f"Train: {train_loss:.4f} | "
              f"Val: {val_loss:.4f} | "
              f"Dice: {val_metrics['dice']:.4f} | "
              f"IoU: {val_metrics['iou']:.4f} | "
              f"LR: {current_lr:.6f} | "
              f"{elapsed/60:.1f}m{marker}")

total_time = time.time() - start_time
print(f"\n{'=' * 70}")
print(f"Egitim tamamlandi! Sure: {total_time/60:.1f} dakika")
print(f"En iyi Val Dice: {best_val_dice:.4f}")

---
## 7. Eğitim Grafikleri

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Val', linewidth=2)
axes[0].set_title('Loss (DiceBCE)', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Dice & IoU
axes[1].plot(history['val_dice'], label='Dice', linewidth=2, color='#e74c3c')
axes[1].plot(history['val_iou'], label='IoU', linewidth=2, color='#3498db')
axes[1].set_title('Validasyon Metrikleri', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

# Learning Rate
axes[2].plot(history['lr'], linewidth=2, color='#2ecc71')
axes[2].set_title('Learning Rate (Cosine)', fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ct_seg_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Doğrulama Sonuçları Görselleştirme

In [ ]:
# En iyi modeli yukle
checkpoint = torch.load(OUTPUT_DIR / 'ct_seg_best_model.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"En iyi model yuklendi (Epoch {checkpoint['epoch']+1}, Dice={checkpoint['val_dice']:.4f})")

# 8 ornek tahmin gorsellestir
fig, axes = plt.subplots(3, 8, figsize=(24, 9))
fig.suptitle(f'Validasyon Tahminleri — Best Dice: {checkpoint["val_dice"]:.4f}', fontsize=14)

indices = np.linspace(0, len(val_dataset) - 1, 8, dtype=int)

with torch.no_grad():
    for col, idx in enumerate(indices):
        img, gt_mask = val_dataset[idx]
        img_gpu = img.unsqueeze(0).to(device)
        
        with torch.amp.autocast('cuda'):
            pred = model(img_gpu)
        pred_mask = (torch.sigmoid(pred) > 0.5).cpu().numpy()[0, 0]
        
        # Denormalize
        img_np = img.numpy().transpose(1, 2, 0)
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img_np = (img_np * std + mean).clip(0, 1)
        
        gt_np = gt_mask.numpy()[0]
        
        # Goruntu
        axes[0, col].imshow(img_np[:, :, 0], cmap='gray')
        axes[0, col].set_title(f'Dilim {idx}', fontsize=9)
        axes[0, col].axis('off')
        
        # Ground Truth
        axes[1, col].imshow(img_np[:, :, 0], cmap='gray')
        if gt_np.max() > 0:
            axes[1, col].contour(gt_np, colors='lime', linewidths=1.5, levels=[0.5])
        axes[1, col].set_title('GT', fontsize=9)
        axes[1, col].axis('off')
        
        # Prediction
        axes[2, col].imshow(img_np[:, :, 0], cmap='gray')
        if pred_mask.max() > 0:
            axes[2, col].contour(pred_mask, colors='red', linewidths=1.5, levels=[0.5])
        axes[2, col].set_title('Tahmin', fontsize=9)
        axes[2, col].axis('off')

axes[0, 0].set_ylabel('Goruntu', fontsize=11)
axes[1, 0].set_ylabel('GT Maske', fontsize=11)
axes[2, 0].set_ylabel('Tahmin', fontsize=11)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ct_seg_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Connected Component Analizi — Nodül Çıkarımı

In [ ]:
from scipy import ndimage

def extract_nodules_from_mask(pred_mask, min_area=10):
    """Binary maskeden bireysel nodulleri cikar.
    
    Args:
        pred_mask: (H, W) binary maske
        min_area: Minimum piksel alani (kucuk gurultu filtreleme)
    
    Returns:
        nodules: [{'center': (y, x), 'area': int, 'bbox': (y1, x1, y2, x2), 'diameter_px': float}]
    """
    labeled, n_components = ndimage.label(pred_mask)
    nodules = []
    
    for i in range(1, n_components + 1):
        component = (labeled == i)
        area = component.sum()
        
        if area < min_area:
            continue
        
        # Merkez
        center = ndimage.center_of_mass(component)
        
        # Bounding box
        rows = np.any(component, axis=1)
        cols = np.any(component, axis=0)
        y1, y2 = np.where(rows)[0][[0, -1]]
        x1, x2 = np.where(cols)[0][[0, -1]]
        
        # Yaklasik cap
        diameter = np.sqrt(area / np.pi) * 2
        
        nodules.append({
            'center': (float(center[0]), float(center[1])),
            'area': int(area),
            'bbox': (int(y1), int(x1), int(y2), int(x2)),
            'diameter_px': float(diameter),
        })
    
    return nodules

# Test: validasyon seti uzerinde nodul cikari
total_nodules_found = 0
total_gt_nodules = 0

with torch.no_grad():
    for i in range(min(100, len(val_dataset))):
        img, gt_mask = val_dataset[i]
        img_gpu = img.unsqueeze(0).to(device)
        
        with torch.amp.autocast('cuda'):
            pred = model(img_gpu)
        pred_mask = (torch.sigmoid(pred) > 0.5).cpu().numpy()[0, 0]
        gt_np = gt_mask.numpy()[0]
        
        pred_nodules = extract_nodules_from_mask(pred_mask)
        gt_nodules = extract_nodules_from_mask(gt_np)
        
        total_nodules_found += len(pred_nodules)
        total_gt_nodules += len(gt_nodules)

print(f"Connected Component Analizi (ilk 100 dilim):")
print(f"  GT nodul sayisi: {total_gt_nodules}")
print(f"  Tahmin edilen: {total_nodules_found}")
if total_gt_nodules > 0:
    print(f"  Tespit orani: {min(total_nodules_found, total_gt_nodules) / total_gt_nodules * 100:.1f}%")

---
## 10. Final Değerlendirme ve Kaydetme

In [ ]:
# Tam validasyon seti uzerinde final metrikler
model.eval()
all_dices = []
all_ious = []
all_precisions = []
all_recalls = []

with torch.no_grad():
    for images, masks in val_loader:
        images = images.to(device)
        masks = masks.to(device)
        
        with torch.amp.autocast('cuda'):
            outputs = model(images)
        
        metrics = compute_metrics(outputs, masks)
        all_dices.append(metrics['dice'])
        all_ious.append(metrics['iou'])
        all_precisions.append(metrics['precision'])
        all_recalls.append(metrics['recall'])

print("=" * 60)
print("FINAL VALIDASYON SONUCLARI")
print("=" * 60)
print(f"  Dice Score:  {np.mean(all_dices):.4f} +/- {np.std(all_dices):.4f}")
print(f"  IoU:         {np.mean(all_ious):.4f} +/- {np.std(all_ious):.4f}")
print(f"  Precision:   {np.mean(all_precisions):.4f} +/- {np.std(all_precisions):.4f}")
print(f"  Recall:      {np.mean(all_recalls):.4f} +/- {np.std(all_recalls):.4f}")

# Metrikleri CSV kaydet
metrics_df = pd.DataFrame({
    'metric': ['dice', 'iou', 'precision', 'recall'],
    'mean': [np.mean(all_dices), np.mean(all_ious), np.mean(all_precisions), np.mean(all_recalls)],
    'std': [np.std(all_dices), np.std(all_ious), np.std(all_precisions), np.std(all_recalls)],
})
metrics_df.to_csv(OUTPUT_DIR / 'ct_seg_metrics.csv', index=False)
print(f"\nMetrikler kaydedildi: ct_seg_metrics.csv")

# History CSV
history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / 'ct_seg_training_history.csv', index=False)
print(f"Egitim historisi: ct_seg_training_history.csv")

# Model boyutu
model_path = OUTPUT_DIR / 'ct_seg_best_model.pth'
print(f"\nModel dosyasi: {model_path.stat().st_size / 1e6:.1f} MB")

In [ ]:
# Ozet rapor
print("\n" + "=" * 65)
print("AlpCAN CT Pipeline — Nodul Segmentasyon Egitimi OZET")
print("=" * 65)

print(f"\n--- Dataset ---")
print(f"  LIDC-IDRI (zhangweiled/lidcidri)")
print(f"  Train: {len(train_df):,} dilim ({len(train_patients)} hasta)")
print(f"  Val:   {len(val_df):,} dilim ({len(val_patients)} hasta)")
print(f"  Maske: Konsensus (>= 2/4 annotator)")

print(f"\n--- Model ---")
print(f"  Mimari: U-Net (ResNet-34 encoder, ImageNet pre-trained)")
print(f"  Giris: {IMG_SIZE}x{IMG_SIZE}x3")
print(f"  Parametre: {n_params:,} ({n_params/1e6:.1f}M)")
print(f"  Loss: DiceBCE (0.5 + 0.5)")
print(f"  Optimizer: AdamW (lr=1e-4)")
print(f"  Epoch: {NUM_EPOCHS}")

print(f"\n--- Performans ---")
print(f"  Val Dice:  {np.mean(all_dices):.4f}")
print(f"  Val IoU:   {np.mean(all_ious):.4f}")

print(f"\n--- Cikti Dosyalari ---")
for f in sorted(OUTPUT_DIR.glob('ct_seg_*')):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name} ({size_mb:.1f} MB)")

print(f"\n--- Sonraki Adimlar ---")
print(f"  1. Model agirliklarini AlpCAN backend'e entegre et")
print(f"  2. nodule_detection_inference.py guncelle")
print(f"  3. Nodul karakterizasyon modeli egit (Notebook 07)")
print(f"  4. 3D volumetrik segmentasyon (tam BT serisi)")
print("=" * 65)